In [0]:
from pyspark.sql.functions import *

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
df = spark.table('customer360_cata.bronze.payments')

In [0]:
df =df.drop("_rescued_data")

In [0]:
df = df.withColumn(
    "payment_date",
    coalesce(
        try_to_date(col("payment_date"), "yyyy/MM/dd"),
        try_to_date(col("payment_date"), "dd-MM-yyyy"),
        try_to_date(col("payment_date"), "yyyy-MM-dd"),
        try_to_date(col("payment_date"), "yyyyMMdd"),
        try_to_date(col("payment_date"), "MM-dd-yyyy"),
        try_to_date(col("payment_date"), "dd/MM/yyyy"),
    )
)

In [0]:
payment_types_df = spark.table('customer360_cata.silver.payment_types')

In [0]:
df = df.join(payment_types_df.alias("p"), df.payment_method == col("p.payment_id"), "inner") \
    .select(df.payment_id, df.customer_id, df.payment_date, col("p.payment_method"), df.payment_status, df.amount)

In [0]:
df =df.withColumn('payment_method',initcap(col("payment_method")))\

df = df.withColumn('payment_status',initcap(when(col("payment_status").isNull(),"Failed").otherwise(col("payment_status"))))

df = df.withColumn("amount",col("amount").cast("double"))
df = df.withColumn('amount',when(col('amount').isNull(),0).when(col('amount')<0,0).otherwise(col('amount')))

In [0]:
df =df.dropDuplicates(["payment_id","customer_id"])
df = df.dropna(subset=["payment_id","customer_id"])

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int"))

In [0]:
df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true") \
    .option('path',base_adls2_path + '/silver/payments') \
        .saveAsTable('customer360_cata.silver.payments')